# Penyelarasan Data Temporal — Berita Geopolitik & Kurs USD/IDR

Notebook ini menyelaraskan data berita geopolitik (`geopolitical_news.csv`, hasil `00a_scraper.ipynb`) dengan data kurs harian USD/IDR dari Bank Indonesia (`jisdor_usd_idr.csv`, hasil `00c_timeseries_jisdor.ipynb`) berdasarkan waktu terbit beritanya.

## 0. Import Library

In [1]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", 80)

## 1. Latar Belakang Masalah

Dua sumber data yang kita punya punya karakteristik waktu yang berbeda:

- **Data kurs (JISDOR)** hanya tersedia di hari kerja pasar sehingga tidak ada nilai di akhir pekan atau hari libur nasional.
- **Data berita** bisa terbit kapan saja, 24 jam, termasuk akhir pekan dan hari libur.

Akibatnya, sebuah berita yang terbit di hari Sabtu, Minggu, hari libur, atau setelah jam tutup pasar pada hari kerja, tidak punya "pasangan" langsung di data kurs pada tanggal yang sama persis. Notebook ini menetapkan aturan logis untuk memetakan tiap berita ke hari trading yang paling relevan, tanpa membuang informasinya.

## Load Data

In [2]:
DATA_DIR = Path("../data")

import os
# "full"     = articles_clean.csv (B_1)
# "filtered" = articles_filtered.csv (B_2 dan C)
# "id"       = B_3: artikel Inggris terfilter (articles_filtered.csv) + artikel CNBC Indonesia (articles_id_clean.csv),
#              satu baris per hari trading dengan kolom teks terpisah (_en dan _id)
CORPUS = "id"
assert CORPUS in ("full", "filtered", "id")
OUT_FILE = {"full": "aligned_dataset.csv", "filtered": "aligned_dataset_filtered.csv", "id": "aligned_dataset_id.csv"}[CORPUS]

kurs = pd.read_csv(DATA_DIR / "processed" / "jisdor_usd_idr.csv", parse_dates=["date"])
kurs = kurs.sort_values("date").reset_index(drop=True)

if CORPUS == "id":
    KOLOM = ["published_at", "clean_title", "lead_paragraph", "clean_content", "url"]
    news_en = pd.read_csv(DATA_DIR / "processed" / "articles_filtered.csv", parse_dates=["published_at"])[KOLOM].assign(lang="en")
    news_id = pd.read_csv(DATA_DIR / "processed" / "articles_id_clean.csv", parse_dates=["published_at"])[KOLOM].assign(lang="id")
    news = pd.concat([news_en, news_id], ignore_index=True)
    print(f"Berita Inggris   : {len(news_en)} baris")
    print(f"Berita Indonesia : {len(news_id)} baris")
else:
    ART_FILE = "articles_clean.csv" if CORPUS == "full" else "articles_filtered.csv"
    news = pd.read_csv(DATA_DIR / "processed" / ART_FILE, parse_dates=["published_at"])  # sudah dibersihkan 00b_preprocessing.ipynb
    news["lang"] = "en"

print(f"Data kurs   : {len(kurs)} baris, {kurs['date'].min().date()} s.d. {kurs['date'].max().date()}")
print(f"Data berita : {len(news)} baris, {news['published_at'].min()} s.d. {news['published_at'].max()}")

kurs.head()

Berita Inggris   : 11764 baris
Berita Indonesia : 10075 baris
Data kurs   : 1202 baris, 2021-09-01 s.d. 2026-09-01
Data berita : 21839 baris, 2021-09-01 06:29:01 s.d. 2026-08-31 07:00:00


,date,usd_idr
0,2021-09-01,14284
1,2021-09-02,14281
2,2021-09-03,14261
3,2021-09-06,14239
4,2021-09-07,14195


In [3]:
news.head()

,published_at,clean_title,lead_paragraph,clean_content,url,lang
0,2021-09-01 07:00:00,Weekly mortgage-refinance demand drops as interest rates stall,A prolonged period of low mortgage rates is taking its toll on the refinance...,A prolonged period of low mortgage rates is taking its toll on the refinance...,https://www.cnbc.com/2021/09/01/weekly-mortgage-refinance-demand-drops-as-in...,en
1,2021-09-01 07:00:00,"U.S. relationship with Taliban unclear after end of Afghanistan War, senior ...",Secretary of Defense Lloyd Austin said Wednesday it was not yet clear what k...,Secretary of Defense Lloyd Austin said Wednesday it was not yet clear what k...,https://www.cnbc.com/2021/09/01/afghanistan-update-us-relationship-with-tali...,en
2,2021-09-01 07:00:00,"China's health-care sector could be Beijing's next regulatory target, analys...","China's health-care sector will probably be the next to fall under scrutiny,...","China's health-care sector will probably be the next to fall under scrutiny,...",https://www.cnbc.com/2021/09/02/chinas-regulatory-crackdown-on-healthcare-se...,en
3,2021-09-02 07:00:00,Joe Manchin voted to advance the $3.5 trillion budget bill — now he says Dem...,Sen. Joe Manchin just made it clear Democrats have work to do to win his vot...,Sen. Joe Manchin just made it clear Democrats have work to do to win his vot...,https://www.cnbc.com/2021/09/02/joe-manchin-opposes-3point5-trillion-biden-d...,en
4,2021-09-02 07:00:00,"China cracks down on showbiz for 'polluting' society and youth, bars 'effemi...","China ordered broadcasters on Thursday to shun artists with ""incorrect polit...","China ordered broadcasters on Thursday to shun artists with ""incorrect polit...",https://www.cnbc.com/2021/09/02/china-cracks-down-on-showbiz-for-polluting-s...,en


## 3. Konversi Zona Waktu

Kolom `published_at` berasal dari `pubDate` RSS Google News, yang formatnya GMT (=UTC). Data kurs JISDOR mengikuti jam operasional pasar Indonesia (WIB, UTC+7). Supaya aturan "jam berapa market tutup" bisa diterapkan dengan benar, kedua data ini perlu disamakan zona waktunya dulu, kita konversi timestamp berita dari UTC ke WIB.

In [4]:
# Artikel Inggris: published_at berformat UTC -> dikonversi ke WIB.
# Artikel CNBC Indonesia: waktu terbit (dari URL) sudah WIB, jadi tidak boleh dikonversi dari UTC lagi.
is_id = news["lang"] == "id"
wib_id = news.loc[is_id, "published_at"].dt.tz_localize("Asia/Jakarta")
wib_en = news.loc[~is_id, "published_at"].dt.tz_localize("UTC").dt.tz_convert("Asia/Jakarta")
news["published_wib"] = pd.concat([wib_en, wib_id]).sort_index()

news[["published_at", "published_wib"]].head()

,published_at,published_wib
0,2021-09-01 07:00:00,2021-09-01 14:00:00+07:00
1,2021-09-01 07:00:00,2021-09-01 14:00:00+07:00
2,2021-09-01 07:00:00,2021-09-01 14:00:00+07:00
3,2021-09-02 07:00:00,2021-09-02 14:00:00+07:00
4,2021-09-02 07:00:00,2021-09-02 14:00:00+07:00


## 4. Aturan Penyelarasan Temporal

Aturan yang digunakan (roll-forward ke hari trading berikutnya):

1. Daftar "hari trading valid" diambil langsung dari tanggal-tanggal yang ada di data kurs JISDOR, jadi hari libur nasional maupun akhir pekan otomatis terdeteksi tanpa perlu kalender libur terpisah, karena BI memang tidak mempublikasikan kurs pada hari-hari tersebut.
2. Ditetapkan jam cutoff 16:00 WIB sebagai asumsi jam tutup pasar valas domestik. Berita yang terbit sebelum jam ini pada hari kerja dianggap mempengaruhi kurs hari itu juga.
3. Berita yang terbit **setelah** jam cutoff, atau pada akhir pekan/hari libur, digeser maju ke hari trading valid berikutnya.
4. Berita tidak dihapus, hanya dipetakan ulang ke tanggal yang paling masuk akal secara logis, supaya sinyalnya tetap terpakai untuk pengujian hipotesis di tahap selanjutnya.

In [5]:
CUTOFF_JAM = 16  # asumsi: pasar valas domestik dianggap tutup jam 16:00 WIB

trading_days = pd.DatetimeIndex(kurs["date"].dt.normalize().unique())


def geser_ke_trading_day(ts, trading_days=trading_days, cutoff=CUTOFF_JAM):
    """Memetakan satu timestamp berita ke hari trading valid berikutnya."""
    tanggal = ts.normalize().tz_localize(None)
    if ts.hour >= cutoff:
        tanggal += pd.Timedelta(days=1)
    while tanggal not in trading_days:
        tanggal += pd.Timedelta(days=1)
        if tanggal > trading_days.max() + pd.Timedelta(days=10):
            return pd.NaT  # di luar cakupan data kurs
    return tanggal


news["trading_day"] = news["published_wib"].apply(geser_ke_trading_day)

n_gagal = news["trading_day"].isna().sum()
print(f"Total berita         : {len(news)}")
print(f"Berhasil dipetakan   : {len(news) - n_gagal}")
print(f"Gagal dipetakan      : {n_gagal} (di luar rentang tanggal data kurs)")

Total berita         : 21839
Berhasil dipetakan   : 21839
Gagal dipetakan      : 0 (di luar rentang tanggal data kurs)


## 5. Agregasi Berita per Hari Trading

Karena satu hari trading bisa menampung lebih dari satu berita (dan sebaliknya, satu hari trading bisa tidak punya berita sama sekali karena strategi *sampling* bi-weekly pada tahap scraping), berita perlu diagregasi dulu menjadi satu baris per hari sebelum digabung ke data kurs. Di sini kita hitung jumlah berita, menggabungkan judul-judulnya, dan menggabungkan isi teks bersihnya (clean_content) per hari sehingga hasil akhir bisa langsung dipakai untuk ekstraksi fitur teks di Tugas 2 tanpa perlu join ulang ke articles_clean.csv

In [6]:
def cap_daily_text(text_series, max_words=1500):
    combined = " | ".join(text_series.dropna())
    words = combined.split()
    return " | ".join(words[:max_words]) if len(words) > max_words else combined


def agregasi(df):
    return df.dropna(subset=["trading_day"]).groupby("trading_day").agg(
        jumlah_berita=("url", "count"),
        judul_gabungan=("clean_title", lambda x: " | ".join(x)),
        lead_paragraph_gabungan=("lead_paragraph", lambda x: cap_daily_text(x, max_words=1500)),
        konten_gabungan=("clean_content", lambda x: "\n\n---\n\n".join(x)),
    ).reset_index()


if CORPUS == "id":
    # tiap bahasa diagregasi sendiri, lalu digabung per hari dengan akhiran _en dan _id
    agg = None
    for lg in ("en", "id"):
        a = agregasi(news[news["lang"] == lg])
        a = a.rename(columns={c: f"{c}_{lg}" for c in a.columns if c != "trading_day"})
        agg = a if agg is None else agg.merge(a, on="trading_day", how="outer")
else:
    agg = agregasi(news)

agg.head()

,trading_day,jumlah_berita_en,judul_gabungan_en,lead_paragraph_gabungan_en,konten_gabungan_en,jumlah_berita_id,judul_gabungan_id,lead_paragraph_gabungan_id,konten_gabungan_id
0,2021-09-01,3.0,Weekly mortgage-refinance demand drops as interest rates stall | U.S. relati...,A prolonged period of low mortgage rates is taking its toll on the refinance...,A prolonged period of low mortgage rates is taking its toll on the refinance...,6.0,"Awas Panas! China Buat Aturan Sepihak di Laut China Selatan | Selow, Inflasi...","China mulai memberlakukan aturan barunya di Laut China Selatan hari ini, Rab...","China mulai memberlakukan aturan barunya di Laut China Selatan hari ini, Rab..."
1,2021-09-02,2.0,Joe Manchin voted to advance the $3.5 trillion budget bill — now he says Dem...,Sen. Joe Manchin just made it clear Democrats have work to do to win his vot...,Sen. Joe Manchin just made it clear Democrats have work to do to win his vot...,2.0,"Inflasi RI Masih Moderat, Yield Mayoritas SBN Kembali Turun | Covid Masih 'M...","Jakarta, CNBC Indonesia - Harga mayoritas obligasi pemerintah atau Surat Ber...","Jakarta, CNBC Indonesia - Harga mayoritas obligasi pemerintah atau Surat Ber..."
2,2021-09-03,4.0,"Jobs report disappoints — only 235,000 positions added vs. expectations of 7...","Job creation for August was a huge disappointment, with the economy adding j...","Job creation for August was a huge disappointment, with the economy adding j...",NaN,NaN,NaN,NaN
3,2021-09-06,6.0,"'Stagflation' is the greatest threat to Europe’s recovery, warns ex-Italian ...",Former Italian Prime Minister Mario Monti told CNBC Saturday that he believe...,Former Italian Prime Minister Mario Monti told CNBC Saturday that he believe...,2.0,"AS Diterjang Badai Ida, Joe Biden Terjun ke Lokasi! | Jreng.. Biden Minta Ka...","Joe Biden mengunjungi area terdampak Badai Ida di Louisiana, Amerika Serikat...","Joe Biden mengunjungi area terdampak Badai Ida di Louisiana, Amerika Serikat..."
4,2021-09-07,5.0,"Malaysia will start treating Covid as 'endemic' around end-October, trade mi...",Malaysia will start treating Covid-19 as an endemic disease around the end o...,Malaysia will start treating Covid-19 as an endemic disease around the end o...,4.0,Mencekam! Begini Suasana Guinea Usai Militer Kudeta Presiden | Cadangan Devi...,Junta Militer menutup perbatasan di Guinea. Junta militer Guinea menutup per...,Junta Militer menutup perbatasan di Guinea. Junta militer Guinea menutup per...


## 6. Penggabungan dengan Data Kurs 

In [7]:
final = kurs.merge(agg, left_on="date", right_on="trading_day", how="left").drop(columns=["trading_day"])
for c in final.columns[2:]:                      # kolom setelah date dan usd_idr
    if c.startswith("jumlah_berita"):
        final[c] = final[c].fillna(0).astype(int)
    else:
        final[c] = final[c].fillna("")

final.head(10)

,date,usd_idr,jumlah_berita_en,judul_gabungan_en,lead_paragraph_gabungan_en,konten_gabungan_en,jumlah_berita_id,judul_gabungan_id,lead_paragraph_gabungan_id,konten_gabungan_id
0,2021-09-01,14284,3,Weekly mortgage-refinance demand drops as interest rates stall | U.S. relati...,A prolonged period of low mortgage rates is taking its toll on the refinance...,A prolonged period of low mortgage rates is taking its toll on the refinance...,6,"Awas Panas! China Buat Aturan Sepihak di Laut China Selatan | Selow, Inflasi...","China mulai memberlakukan aturan barunya di Laut China Selatan hari ini, Rab...","China mulai memberlakukan aturan barunya di Laut China Selatan hari ini, Rab..."
1,2021-09-02,14281,2,Joe Manchin voted to advance the $3.5 trillion budget bill — now he says Dem...,Sen. Joe Manchin just made it clear Democrats have work to do to win his vot...,Sen. Joe Manchin just made it clear Democrats have work to do to win his vot...,2,"Inflasi RI Masih Moderat, Yield Mayoritas SBN Kembali Turun | Covid Masih 'M...","Jakarta, CNBC Indonesia - Harga mayoritas obligasi pemerintah atau Surat Ber...","Jakarta, CNBC Indonesia - Harga mayoritas obligasi pemerintah atau Surat Ber..."
2,2021-09-03,14261,4,"Jobs report disappoints — only 235,000 positions added vs. expectations of 7...","Job creation for August was a huge disappointment, with the economy adding j...","Job creation for August was a huge disappointment, with the economy adding j...",0,,,
3,2021-09-06,14239,6,"'Stagflation' is the greatest threat to Europe’s recovery, warns ex-Italian ...",Former Italian Prime Minister Mario Monti told CNBC Saturday that he believe...,Former Italian Prime Minister Mario Monti told CNBC Saturday that he believe...,2,"AS Diterjang Badai Ida, Joe Biden Terjun ke Lokasi! | Jreng.. Biden Minta Ka...","Joe Biden mengunjungi area terdampak Badai Ida di Louisiana, Amerika Serikat...","Joe Biden mengunjungi area terdampak Badai Ida di Louisiana, Amerika Serikat..."
4,2021-09-07,14195,5,"Malaysia will start treating Covid as 'endemic' around end-October, trade mi...",Malaysia will start treating Covid-19 as an endemic disease around the end o...,Malaysia will start treating Covid-19 as an endemic disease around the end o...,4,Mencekam! Begini Suasana Guinea Usai Militer Kudeta Presiden | Cadangan Devi...,Junta Militer menutup perbatasan di Guinea. Junta militer Guinea menutup per...,Junta Militer menutup perbatasan di Guinea. Junta militer Guinea menutup per...
5,2021-09-08,14266,6,BlackRock responds to George Soros' criticism over China investments | Job o...,"BlackRock, the world's largest asset manager, has responded to sharp critici...","BlackRock, the world's largest asset manager, has responded to sharp critici...",1,"Tiba-tiba Xi Jinping Bikin Bursa Saham di Beijing, buat Apa?",Presiden China Xi Jinping pada hari Kamis pekan lalu menyatakan negaranya ak...,Presiden China Xi Jinping pada hari Kamis pekan lalu menyatakan negaranya ak...
6,2021-09-09,14272,6,China's rumored ambitions to dive into Afghanistan are overstated and unreal...,One of the first things many Western pundits predicted as the chaotic Americ...,One of the first things many Western pundits predicted as the chaotic Americ...,0,,,
7,2021-09-10,14225,4,Rudy Giuliani associate Igor Fruman pleads guilty to soliciting foreign camp...,"Igor Fruman, a former associate of embattled lawyer Rudy Giuliani, pleaded g...","Igor Fruman, a former associate of embattled lawyer Rudy Giuliani, pleaded g...",4,"Biden Ngamuk! Tiba-tiba Frustrasi Hilang Kesabaran, Ada Apa? | Geger Dunia! ...",Presiden Amerika Serikat (AS) Joe Biden tiba-tiba menunjukkan rasa ketidakse...,Presiden Amerika Serikat (AS) Joe Biden tiba-tiba menunjukkan rasa ketidakse...
8,2021-09-13,14260,12,California's battle with climate change is at stake in Tuesday's recall elec...,California voters will decide whether to remove Democratic Gov. Gavin Newsom...,California voters will decide whether to remove Democratic Gov. Gavin N

## 7. Validasi & Ringkasan Hasil

In [8]:
total_hari = len(final)
print(f"Total hari trading                : {total_hari}")
for jc in [c for c in final.columns if c.startswith("jumlah_berita")]:
    nama = "" if jc == "jumlah_berita" else f" ({jc.split('_')[-1]})"
    hari_ada = (final[jc] > 0).sum()
    print(f"Hari trading dengan >=1 berita{nama:5s}: {hari_ada} ({hari_ada/total_hari:.1%})   tanpa berita: {total_hari - hari_ada}   total berita ter-assign: {final[jc].sum()}")
print(f"Berita di luar cakupan data kurs   : {n_gagal} dari {len(news)} ({n_gagal/len(news):.1%})")
print()

Total hari trading                : 1202
Hari trading dengan >=1 berita (en): 1182 (98.3%)   tanpa berita: 20   total berita ter-assign: 11764
Hari trading dengan >=1 berita (id): 881 (73.3%)   tanpa berita: 321   total berita ter-assign: 10075
Berita di luar cakupan data kurs   : 0 dari 21839 (0.0%)



## 8. Menyimpan Hasil Akhir 

In [9]:
OUTPUT_PATH = DATA_DIR / "processed" / OUT_FILE
final.to_csv(OUTPUT_PATH, index=False)
print(f"Tersimpan di: {OUTPUT_PATH.resolve()}")
print(f"Bentuk akhir: {final.shape[0]} baris x {final.shape[1]} kolom")
final.dtypes

Tersimpan di: C:\UGM Stuff\Semester 5\Natural Language Processing\nlp-geopolitics-usa\data\processed\aligned_dataset_id.csv
Bentuk akhir: 1202 baris x 10 kolom


date                          datetime64[ns]
usd_idr                                int64
jumlah_berita_en                       int64
judul_gabungan_en                     object
lead_paragraph_gabungan_en            object
konten_gabungan_en                    object
jumlah_berita_id                       int64
judul_gabungan_id                     object
lead_paragraph_gabungan_id            object
konten_gabungan_id                    object
dtype: object